In [2]:
import pandas as pd
!pip install dash
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# -------------------------
# Load data
# -------------------------
df = pd.read_csv("Impact_of_Remote_Work_on_Mental_Health.csv")

gender_df = df[df["Gender"].isin(["Male", "Female"])].copy()
gender_df = gender_df.loc[:, ~gender_df.columns.duplicated()]
gender_df = gender_df.dropna(subset=["Industry"])

industries = sorted(gender_df["Industry"].dropna().unique())

numeric_options = {
    "Hours Worked Per Week": "Hours_Worked_Per_Week",
    "Work Life Balance Rating": "Work_Life_Balance_Rating"
}

categorical_options = {
    "Stress Level": "Stress_Level",
    "Access to Mental Health Resources": "Access_to_Mental_Health_Resources",
    "Mental Health Condition": "Mental_Health_Condition",
    "Sleep Quality": "Sleep_Quality"
}

# -------------------------
# Build app
# -------------------------
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Male vs Female Work Habits Dashboard"),

    html.Div([
        html.Label("Select Industry"),
        dcc.Dropdown(
            id="industry-dropdown",
            options=[{"label": i, "value": i} for i in industries],
            value=industries[0],
            clearable=False
        )
    ], style={"width": "48%", "padding": "10px"}),

    html.Div([
        html.Div([
            html.Label("Select Numeric Metric"),
            dcc.Dropdown(
                id="numeric-dropdown",
                options=[{"label": k, "value": v} for k, v in numeric_options.items()],
                value="Hours_Worked_Per_Week",
                clearable=False
            ),
            dcc.Graph(id="numeric-box-plot")
        ], style={"width": "48%", "display": "inline-block", "verticalAlign": "top"}),

        html.Div([
            html.Label("Select Categorical Metric"),
            dcc.Dropdown(
                id="categorical-dropdown",
                options=[{"label": k, "value": v} for k, v in categorical_options.items()],
                value="Stress_Level",
                clearable=False
            ),
            dcc.Graph(id="categorical-bar-chart")
        ], style={"width": "48%", "display": "inline-block", "verticalAlign": "top"})
    ])
], style={"fontFamily": "Arial", "padding": "20px"})

# -------------------------
# Helper
# -------------------------
def get_filtered_data(industry):
    return gender_df[gender_df["Industry"] == industry]

# -------------------------
# Callbacks
# -------------------------
@app.callback(
    Output("numeric-box-plot", "figure"),
    Input("industry-dropdown", "value"),
    Input("numeric-dropdown", "value")
)
def update_numeric_chart(industry, metric):
    filtered = get_filtered_data(industry)

    fig = px.box(
        filtered,
        x="Gender",
        y=metric,
        color="Gender",
        points="all",
        title=f"{metric.replace('_', ' ')} by Gender — {industry}"
    )
    fig.update_layout(template="plotly_white")
    return fig

@app.callback(
    Output("categorical-bar-chart", "figure"),
    Input("industry-dropdown", "value"),
    Input("categorical-dropdown", "value")
)
def update_categorical_chart(industry, metric):
    filtered = get_filtered_data(industry)

    fig = px.histogram(
        filtered,
        x=metric,
        color="Gender",
        barmode="group",
        histnorm="percent",
        title=f"{metric.replace('_', ' ')} by Gender — {industry}"
    )
    fig.update_layout(
        template="plotly_white",
        yaxis_title="Percent"
    )
    return fig

# -------------------------
# Run in Jupyter Notebook
# -------------------------
app.run(debug=False, use_reloader=False, port=8050)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dash]1/2 [dash]
